In [2]:
from pathlib import Path
import polars as pl
from tqdm.auto import tqdm
import re
import json

from gtr_database.parse_opportunity import ParsedOpportunity, Section
from gtr_database.rag_pipelines import RAGPipeline, SimpleChunkingPipeline

In [2]:
DATA_PATH = Path(".").resolve().parent / "data_cache/opportunities"

df = pl.read_ndjson(str(DATA_PATH / "parsed_opportunities.jsonl"))
df

shape: (2_102, 19)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ status ┆ funders    ┆ funding_ty ┆ total_fun ┆ … ┆ href      ┆ co_funder ┆ maximum_a ┆ minimum_a │
│ ---    ┆ ---        ┆ pe         ┆ ding      ┆   ┆ ---       ┆ s         ┆ ward      ┆ ward      │
│ str    ┆ list[str]  ┆ ---        ┆ ---       ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---       │
│        ┆            ┆ str        ┆ str       ┆   ┆           ┆ list[str] ┆ str       ┆ str       │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ £10,000,0 ┆ … ┆ https://w ┆ null      ┆ null      ┆ null      │
│        ┆            ┆            ┆ 00        ┆   ┆ ww.ukri.o ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ rg/opport ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ uni…      ┆           ┆           ┆           │
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ £20,000,0 ┆ … ┆ https://w ┆ null      ┆ null      ┆ null      │
│        ┆            ┆            ┆ 00        ┆   ┆ ww.ukri.o ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ rg/opport ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ uni…      ┆           ┆           ┆           │
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ £1,500,00 ┆ … ┆ https://w ┆ ["The Dep ┆ null      ┆ null      │
│        ┆            ┆            ┆ 0         ┆   ┆ ww.ukri.o ┆ artment   ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ rg/opport ┆ for Trans ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ uni…      ┆ port…     ┆           ┆           │
│ Closed ┆ ["EPSRC"]  ┆ Grant      ┆ £2,000,00 ┆ … ┆ https://w ┆ null      ┆ £250,000  ┆ null      │
│        ┆            ┆            ┆ 0         ┆   ┆ ww.ukri.o ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ rg/opport ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ uni…      ┆           ┆           ┆           │
│ Closed ┆ ["EPSRC"]  ┆ Grant      ┆ £4,000,00 ┆ … ┆ https://w ┆ null      ┆ £1,000,00 ┆ null      │
│        ┆            ┆            ┆ 0         ┆   ┆ ww.ukri.o ┆           ┆ 0         ┆           │
│        ┆            ┆            ┆           ┆   ┆ rg/opport ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ uni…      ┆           ┆           ┆           │
│ …      ┆ …          ┆ …          ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …         │
│ Closed ┆ ["STFC"]   ┆ Grant      ┆ £2,000,00 ┆ … ┆ https://w ┆ null      ┆ null      ┆ null      │
│        ┆            ┆            ┆ 0         ┆   ┆ ww.ukri.o ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ rg/opport ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ uni…      ┆           ┆           ┆           │
│ Closed ┆ ["ESRC",   ┆ Grant      ┆ £3,000,00 ┆ … ┆ https://w ┆ ["Japan   ┆ £425,000  ┆ £350,000  │
│        ┆ "AHRC"]    ┆            ┆ 0         ┆   ┆ ww.ukri.o ┆ Society   ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ rg/opport ┆ for the   ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ uni…      ┆ Promot…   ┆           ┆           │
│ Closed ┆ ["NERC"]   ┆ Grant      ┆ £19,000,0 ┆ … ┆ https://w ┆ null      ┆ £5,000,00 ┆ £2,500,00 │
│        ┆            ┆            ┆ 00        ┆   ┆ ww.ukri.o ┆           ┆ 0         ┆ 0         │
│        ┆            ┆            ┆           ┆   ┆ rg/opport ┆           ┆           ┆           │
│        ┆            ┆            ┆           ┆   ┆ uni…      ┆           ┆           ┆           │
│ Closed ┆ ["NERC"]   ┆ Grant      ┆ £24,0

In [4]:
df["summary"][0]

'See the [full opportunity details on the Innovation Funding Service](https://apply-for-innovation-funding.service.gov.uk/competition/1573/overview/910e2dcf-796d-49e4-bf12-efa1f2be06db).\n\nUK registered businesses can apply for a share of up to £20 million over two rounds of this competition.\n\nThe competition will fund projects focusing on developing digital and data-enabled tools as well as multi-modal approaches.\n\n## Eligibility summary\n\nThis competition is open to collaborations only.\n\nTo lead a project your organisation must be a UK registered:\n\n* business\n* research and technology organisation\n\nYou must be or involve at least one grant claiming [micro, small or medium-sized enterprise](https://www.gov.uk/government/publications/life-of-a-company-annual-requirements/life-of-a-company-part-1-accounts#micro-entity).'

In [3]:
QUESTIONS = {
    "minimum_award":            "What is the minimum fund value (£)?",
    "maximum_award":            "What is the maximum fund value (£)?",
    "total_funding":            "What is the total fund value (£) split among funded applications?",
    "funding_percentage":       "What percentage (%) of the project's full economic cost (FEC) will UKRI fund?",
    "minimum_funding_duration": "What is the minimum duration of the project/funding?",
    "maximum_funding_duration": "What is the maximum duration of the project/funding?",
}

QUERY_VARIANTS = {
    "minimum_award": [
        "What is the minimum fund value?",
        "What is the minimum award amount per project?",
        "What is the smallest grant value available?",
    ],
    "maximum_award": [
        "What is the maximum fund value?",
        "What is the maximum award amount per project?",
        "What is the upper limit or ceiling of the grant?",
    ],
    "total_funding": [
        "What is the total fund value split among successful applications?",
        "What is the total funding available for this opportunity?",
        "What is the total budget or overall pot of money for this call?",
    ],
    "funding_percentage": [
        "What percentage of the project's funding will UKRI fund?",
        "What is the UKRI contribution percentage or fEC rate?",
        "What proportion or share of project costs will UKRI cover?",
    ],
    "minimum_funding_duration": [
        "What is the minimum duration of the project/funding?",
        "What is the minimum project length or term allowed?",
        "What is the shortest period a funded project can last?",
    ],
    "maximum_funding_duration": [
        "What is the maximum duration of the project/funding?",
        "What is the maximum project length or term allowed?",
        "How long can a funded project last?",
    ],
}

EXTRACTION_FIELDS = list(QUESTIONS.keys())

query_samples = (
    df.with_columns(
        funding_percentage=pl.lit(None, dtype=pl.Utf8),
        minimum_funding_duration=pl.lit(None, dtype=pl.Utf8),
        maximum_funding_duration=pl.lit(None, dtype=pl.Utf8),
    )
    .select("id", "title", "summary", "metadata", "sections", "full_text", *EXTRACTION_FIELDS)
    .with_columns(
        questions=pl.concat_list([
            pl.when(pl.col(col).is_null())
              .then(pl.lit(pl.Series([{"field": col, "question": question}])))
              .otherwise(pl.lit(pl.Series([None], dtype=pl.Struct({"field": pl.Utf8, "question": pl.Utf8}))))
            for col, question in QUESTIONS.items()
        ]).list.eval(pl.element().drop_nulls())
    )
    .explode("questions")
    .with_columns(
        target_field=pl.col("questions").struct.field("field"),
        question=pl.col("questions").struct.field("question"),
    )
    .drop("questions", *EXTRACTION_FIELDS)
)
query_samples

shape: (9_938, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ title      ┆ summary    ┆ metadata  ┆ sections  ┆ full_text ┆ target_fi ┆ question  │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ eld       ┆ ---       │
│ str        ┆ str        ┆ str        ┆ str       ┆ list[stru ┆ str       ┆ ---       ┆ str       │
│            ┆            ┆            ┆           ┆ ct[3]]    ┆           ┆ str       ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 64517ad0-c ┆ Funding    ┆ See the    ┆ | Field   ┆ []        ┆ # Funding ┆ minimum_a ┆ What is   │
│ fb3-5d70-8 ┆ opportunit ┆ [full oppo ┆ | Value   ┆           ┆ opportuni ┆ ward      ┆ the       │
│ 904-44d15c ┆ y:         ┆ rtunity    ┆ …         ┆           ┆ ty:       ┆           ┆ minimum   │
│ …          ┆ Advancing… ┆ deta…      ┆           ┆           ┆ Advanci…  ┆           ┆ fund      │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ value…    │
│ 64517ad0-c ┆ Funding    ┆ See the    ┆ | Field   ┆ []        ┆ # Funding ┆ maximum_a ┆ What is   │
│ fb3-5d70-8 ┆ opportunit ┆ [full oppo ┆ | Value   ┆           ┆ opportuni ┆ ward      ┆ the       │
│ 904-44d15c ┆ y:         ┆ rtunity    ┆ …         ┆           ┆ ty:       ┆           ┆ maximum   │
│ …          ┆ Advancing… ┆ deta…      ┆           ┆           ┆ Advanci…  ┆           ┆ fund      │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ value…    │
│ 64517ad0-c ┆ Funding    ┆ See the    ┆ | Field   ┆ []        ┆ # Funding ┆ funding_p ┆ What perc │
│ fb3-5d70-8 ┆ opportunit ┆ [full oppo ┆ | Value   ┆           ┆ opportuni ┆ ercentage ┆ entage    │
│ 904-44d15c ┆ y:         ┆ rtunity    ┆ …         ┆           ┆ ty:       ┆           ┆ (%) of    │
│ …          ┆ Advancing… ┆ deta…      ┆           ┆           ┆ Advanci…  ┆           ┆ the pro…  │
│ 64517ad0-c ┆ Funding    ┆ See the    ┆ | Field   ┆ []        ┆ # Funding ┆ minimum_f ┆ What is   │
│ fb3-5d70-8 ┆ opportunit ┆ [full oppo ┆ | Value   ┆           ┆ opportuni ┆ unding_du ┆ the       │
│ 904-44d15c ┆ y:         ┆ rtunity    ┆ …         ┆           ┆ ty:       ┆ ration    ┆ minimum   │
│ …          ┆ Advancing… ┆ deta…      ┆           ┆           ┆ Advanci…  ┆           ┆ duration  │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ o…        │
│ 64517ad0-c ┆ Funding    ┆ See the    ┆ | Field   ┆ []        ┆ # Funding ┆ maximum_f ┆ What is   │
│ fb3-5d70-8 ┆ opportunit ┆ [full oppo ┆ | Value   ┆           ┆ opportuni ┆ unding_du ┆ the       │
│ 904-44d15c ┆ y:         ┆ rtunity    ┆ …         ┆           ┆ ty:       ┆ ration    ┆ maximum   │
│ …          ┆ Advancing… ┆ deta…      ┆           ┆           ┆ Advanci…  ┆           ┆ duration  │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ o…        │
│ …          ┆ …          ┆ …          ┆ …         ┆ …         ┆ …         ┆ …         ┆ …         │
│ ad9693b7-c ┆ Funding    ┆ Apply for  ┆ | Field   ┆ [{"Who    ┆ # Funding ┆ minimum_a ┆ What is   │
│ db6-5a6b-b ┆ opportunit ┆ funding to ┆ | Value   ┆ can apply ┆ opportuni ┆ ward      ┆ the       │
│ 86d-d1971a ┆ y:         ┆ address o… ┆ …         ┆ ",2,[{nul ┆ ty:       ┆           ┆ minimum   │
│ …          ┆ Addressin… ┆            ┆           ┆ l,nul…    ┆ Address…  ┆           ┆ fund      │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ value…    │
│ ad9693b7-c ┆ Funding    ┆ Apply for  ┆ | Field   ┆ [{"Who    ┆ # Funding ┆ maximum_a ┆ What is   │
│ db6-5a6b-b ┆ opportunit ┆ funding to ┆ | Value   ┆ can apply ┆ opportuni ┆ ward      ┆ the       │
│ 86d-d1971a ┆ y:         ┆ address o… ┆ …         ┆ ",2,[{nul ┆ ty:       ┆           ┆ maximum   │
│ …          ┆ Addressin… ┆            ┆   

In [4]:
def build_dataset(
    test_df: pl.DataFrame,
    use_rag: bool = True,
    simple_chunking: bool = False,
    chunk_size: int = 200,
    chunk_overlap: int = 50,
    alpha: float = 0.5,
    use_reranker: bool = True,
) -> pl.DataFrame:
    """Build a dataset with retrieval contexts for each question row."""
    if not use_rag:
        rag = None
    elif simple_chunking:
        rag = SimpleChunkingPipeline(chunk_size=chunk_size, chunk_overlap=chunk_overlap, alpha=alpha, use_reranker=use_reranker)
    else:
        rag = RAGPipeline(alpha=alpha, use_reranker=use_reranker)

    contexts = []
    settings = {"use_rag": use_rag, "simple_chunking": simple_chunking, "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap, "alpha": alpha, "use_reranker": use_reranker}

    pbar = tqdm(total=test_df.height, desc="Building dataset")
    for (opp_id,), group in test_df.group_by("id", maintain_order=True):
        first_row = group.row(0, named=True)
        full_text = first_row["full_text"]

        if use_rag and rag is not None:
            rag.clear()
            if simple_chunking:
                rag.index_text(full_text)
            else:
                rag.index(ParsedOpportunity(
                    title=first_row["title"],
                    summary=first_row["summary"],
                    sections=[Section.from_dict(s) for s in (first_row["sections"] or [])],
                    metadata=first_row["metadata"],
                ))

        for row in group.iter_rows(named=True):
            field = row["target_field"]
            question = row["question"]
            queries = QUERY_VARIANTS.get(field, [question])
            context = rag.query(queries) if use_rag and rag is not None else full_text
            contexts.append(context)
            pbar.update(1)

    return test_df.with_columns(context=pl.Series(contexts), settings=pl.lit(settings))

# Use best result from experiment
dataset = pl.DataFrame(build_dataset(
    test_df=query_samples,
    use_rag=True,
    simple_chunking=False,
    alpha=0.25,
    use_reranker=False,
))

dataset.write_ndjson(Path(".").parent / "final_extraction" / "dataset_permutations.jsonl")
dataset

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building dataset:   0%|          | 0/9938 [00:00<?, ?it/s]

shape: (9_938, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ title     ┆ summary   ┆ metadata  ┆ … ┆ target_fi ┆ question  ┆ context   ┆ settings │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ eld       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ ---       ┆ str       ┆ str       ┆ struct[6 │
│           ┆           ┆           ┆           ┆   ┆ str       ┆           ┆           ┆ ]        │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 64517ad0- ┆ Funding   ┆ See the   ┆ | Field   ┆ … ┆ minimum_a ┆ What is   ┆ | Field   ┆ {true,fa │
│ cfb3-5d70 ┆ opportuni ┆ [full opp ┆ | Value   ┆   ┆ ward      ┆ the       ┆ | Value   ┆ lse,200, │
│ -8904-44d ┆ ty: Advan ┆ ortunity  ┆ …         ┆   ┆           ┆ minimum   ┆ …         ┆ 50,0.25, │
│ 15c…      ┆ cing…     ┆ deta…     ┆           ┆   ┆           ┆ fund      ┆           ┆ false}   │
│           ┆           ┆           ┆           ┆   ┆           ┆ value…    ┆           ┆          │
│ 64517ad0- ┆ Funding   ┆ See the   ┆ | Field   ┆ … ┆ maximum_a ┆ What is   ┆ | Field   ┆ {true,fa │
│ cfb3-5d70 ┆ opportuni ┆ [full opp ┆ | Value   ┆   ┆ ward      ┆ the       ┆ | Value   ┆ lse,200, │
│ -8904-44d ┆ ty: Advan ┆ ortunity  ┆ …         ┆   ┆           ┆ maximum   ┆ …         ┆ 50,0.25, │
│ 15c…      ┆ cing…     ┆ deta…     ┆           ┆   ┆           ┆ fund      ┆           ┆ false}   │
│           ┆           ┆           ┆           ┆   ┆           ┆ value…    ┆           ┆          │
│ 64517ad0- ┆ Funding   ┆ See the   ┆ | Field   ┆ … ┆ funding_p ┆ What perc ┆ | Field   ┆ {true,fa │
│ cfb3-5d70 ┆ opportuni ┆ [full opp ┆ | Value   ┆   ┆ ercentage ┆ entage    ┆ | Value   ┆ lse,200, │
│ -8904-44d ┆ ty: Advan ┆ ortunity  ┆ …         ┆   ┆           ┆ (%) of    ┆ …         ┆ 50,0.25, │
│ 15c…      ┆ cing…     ┆ deta…     ┆           ┆   ┆           ┆ the pro…  ┆           ┆ false}   │
│ 64517ad0- ┆ Funding   ┆ See the   ┆ | Field   ┆ … ┆ minimum_f ┆ What is   ┆ | Field   ┆ {true,fa │
│ cfb3-5d70 ┆ opportuni ┆ [full opp ┆ | Value   ┆   ┆ unding_du ┆ the       ┆ | Value   ┆ lse,200, │
│ -8904-44d ┆ ty: Advan ┆ ortunity  ┆ …         ┆   ┆ ration    ┆ minimum   ┆ …         ┆ 50,0.25, │
│ 15c…      ┆ cing…     ┆ deta…     ┆           ┆   ┆           ┆ duration  ┆           ┆ false}   │
│           ┆           ┆           ┆           ┆   ┆           ┆ o…        ┆           ┆          │
│ 64517ad0- ┆ Funding   ┆ See the   ┆ | Field   ┆ … ┆ maximum_f ┆ What is   ┆ | Field   ┆ {true,fa │
│ cfb3-5d70 ┆ opportuni ┆ [full opp ┆ | Value   ┆   ┆ unding_du ┆ the       ┆ | Value   ┆ lse,200, │
│ -8904-44d ┆ ty: Advan ┆ ortunity  ┆ …         ┆   ┆ ration    ┆ maximum   ┆ …         ┆ 50,0.25, │
│ 15c…      ┆ cing…     ┆ deta…     ┆           ┆   ┆           ┆ duration  ┆           ┆ false}   │
│           ┆           ┆           ┆           ┆   ┆           ┆ o…        ┆           ┆          │
│ …         ┆ …         ┆ …         ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …        │
│ ad9693b7- ┆ Funding   ┆ Apply for ┆ | Field   ┆ … ┆ minimum_a ┆ What is   ┆ | Field   ┆ {true,fa │
│ cdb6-5a6b ┆ opportuni ┆ funding   ┆ | Value   ┆   ┆ ward      ┆ the       ┆ | Value   ┆ lse,200, │
│ -b86d-d19 ┆ ty: Addre ┆ to        ┆ …         ┆   ┆           ┆ minimum   ┆ …         ┆ 50,0.25, │
│ 71a…      ┆ ssin…     ┆ address   ┆           ┆   ┆           ┆ fund      ┆           ┆ false}   │
│           ┆           ┆ o…        ┆           ┆   ┆           ┆ value…    ┆           ┆          │
│ ad9693b7- ┆ Funding   ┆ Apply for ┆ | Field   ┆ … ┆ maximum_a ┆ What is   ┆ | Field   ┆ {true,fa │
│ cdb6-5a6b ┆ opportuni ┆ funding   ┆ | Value   ┆   ┆ ward      ┆ the       ┆ | Value   ┆ lse,200, │
│ -b86d-d19 ┆ ty: Addre ┆ to        ┆ …         ┆   ┆           ┆ maximum   ┆ …         ┆ 50,0.25, │
│ 71a…      ┆ ssin…     ┆ address   ┆     

In [19]:
def _extract_answer(content: str | None) -> str | None:
    """Robustly extract the answer from a model response."""
    if content is None:
        return None
    c = content.strip().removeprefix("```json").removesuffix("```").strip()
    try:
        obj = json.loads(c)
    except json.JSONDecodeError:
        # Some responses use commas instead of colons: {"key", "value"}
        # Try replacing first comma with colon
        c2 = c.replace(",", ":", 1)
        try:
            obj = json.loads(c2)
        except json.JSONDecodeError:
            return None
    if isinstance(obj, dict):
        # Get the first (and usually only) value regardless of key name
        val = next(iter(obj.values()), None)
        if val is None:
            return None
        text = str(val).strip()
        return None if text.upper() == "UNKNOWN" else text
    return None

query_responses = pl.read_ndjson(str(Path(".").parent / "final_extraction" / "dataset_permutations_results.jsonl"))

responses = [
    _extract_answer(row["_response"]["choices"][0]["message"]["content"])
    for row in query_responses.select("_response").to_dicts()
]

query_responses = query_responses.with_columns(responses=pl.Series(responses))

In [21]:
extracted = (
    query_responses
    .select("id", "target_field", "responses")
    .pivot(on="target_field", index="id", values="responses", aggregate_function="first")
)

result = df.join(extracted, on="id", how="left", suffix="_extracted")

for field in EXTRACTION_FIELDS:
    orig = field in df.columns
    extr = f"{field}_extracted" in result.columns
    if orig and extr:
        result = result.with_columns(
            pl.coalesce(pl.col(field), pl.col(f"{field}_extracted")).alias(field)
        ).drop(f"{field}_extracted")
    elif extr:
        result = result.rename({f"{field}_extracted": field})

result

shape: (2_102, 22)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ status ┆ funders    ┆ funding_ty ┆ total_fun ┆ … ┆ minimum_a ┆ minimum_f ┆ maximum_f ┆ funding_p │
│ ---    ┆ ---        ┆ pe         ┆ ding      ┆   ┆ ward      ┆ unding_du ┆ unding_du ┆ ercentage │
│ str    ┆ list[str]  ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ration    ┆ ration    ┆ ---       │
│        ┆            ┆ str        ┆ str       ┆   ┆ str       ┆ ---       ┆ ---       ┆ str       │
│        ┆            ┆            ┆           ┆   ┆           ┆ str       ┆ str       ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ £10,000,0 ┆ … ┆ 10000000  ┆ null      ┆ null      ┆ null      │
│        ┆            ┆            ┆ 00        ┆   ┆           ┆           ┆           ┆           │
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ £20,000,0 ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│        ┆            ┆            ┆ 00        ┆   ┆           ┆           ┆           ┆           │
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ £1,500,00 ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│        ┆            ┆            ┆ 0         ┆   ┆           ┆           ┆           ┆           │
│ Closed ┆ ["EPSRC"]  ┆ Grant      ┆ £2,000,00 ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│        ┆            ┆            ┆ 0         ┆   ┆           ┆           ┆           ┆           │
│ Closed ┆ ["EPSRC"]  ┆ Grant      ┆ £4,000,00 ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│        ┆            ┆            ┆ 0         ┆   ┆           ┆           ┆           ┆           │
│ …      ┆ …          ┆ …          ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …         │
│ Closed ┆ ["STFC"]   ┆ Grant      ┆ £2,000,00 ┆ … ┆ null      ┆ null      ┆ three     ┆ null      │
│        ┆            ┆            ┆ 0         ┆   ┆           ┆           ┆ years     ┆           │
│ Closed ┆ ["ESRC",   ┆ Grant      ┆ £3,000,00 ┆ … ┆ £350,000  ┆ 3 years   ┆ 3 years   ┆ 80        │
│        ┆ "AHRC"]    ┆            ┆ 0         ┆   ┆           ┆           ┆           ┆           │
│ Closed ┆ ["NERC"]   ┆ Grant      ┆ £19,000,0 ┆ … ┆ £2,500,00 ┆ null      ┆ 4 years   ┆ 80        │
│        ┆            ┆            ┆ 00        ┆   ┆ 0         ┆           ┆           ┆           │
│ Closed ┆ ["NERC"]   ┆ Grant      ┆ £24,000,0 ┆ … ┆ £3,000,00 ┆ null      ┆ 4 years   ┆ 80        │
│        ┆            ┆            ┆ 00        ┆   ┆ 0         ┆           ┆           ┆           │
│ Closed ┆ ["NERC"]   ┆ Grant      ┆ £20,000,0 ┆ … ┆ null      ┆ null      ┆ four      ┆ 80        │
│        ┆            ┆            ┆ 00        ┆   ┆           ┆           ┆ years     ┆           │
└────────┴────────────┴────────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

In [ ]:
WORD_TO_NUM = {
    "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "eleven": 11, "twelve": 12, "fifteen": 15, "eighteen": 18,
    "twenty": 20, "twenty-four": 24, "thirty": 30, "thirty-six": 36,
    "forty": 40, "forty-eight": 48, "fifty": 50, "sixty": 60,
}


def parse_pounds(text: str | None) -> int | None:
    """Parse a monetary value to integer pounds."""
    if text is None:
        return None
    text = str(text).replace(",", "").replace(" ", "").strip()
    if not text:
        return None
    try:
        return int(float(text))
    except ValueError:
        pass
    m = re.search(r"(?:£|Â£)?([\d.]+)\s*(million|m|billion|b|thousand|k)?", text, re.IGNORECASE)
    if m:
        value = float(m.group(1))
        unit = (m.group(2) or "").lower()
        multipliers = {"million": 1e6, "m": 1e6, "billion": 1e9, "b": 1e9, "thousand": 1e3, "k": 1e3}
        value *= multipliers.get(unit, 1)
        return int(value)
    return None


def parse_percentage(text: str | None) -> int | None:
    """Parse a percentage to integer."""
    if text is None:
        return None
    m = re.search(r"(\d+)\s*%?", str(text).strip())
    return int(m.group(1)) if m else None


def parse_duration_days(text: str | None) -> int | None:
    """Parse a duration string to integer days."""
    if text is None:
        return None
    text = str(text).lower().strip()
    if not text:
        return None
    for word, num in WORD_TO_NUM.items():
        text = re.sub(rf"\b{word}\b", str(num), text)
    total_days = 0
    for m in re.finditer(r"(\d+)\s*(year|month|week|day)s?", text):
        n = int(m.group(1))
        unit = m.group(2)
        if unit == "year":     total_days += n * 365
        elif unit == "month":  total_days += n * 30
        elif unit == "week":   total_days += n * 7
        elif unit == "day":    total_days += n
    return total_days if total_days > 0 else None


FIELD_PARSERS = {
    "minimum_award": parse_pounds,
    "maximum_award": parse_pounds,
    "total_funding": parse_pounds,
    "funding_percentage": parse_percentage,
    "minimum_funding_duration": parse_duration_days,
    "maximum_funding_duration": parse_duration_days,
}


def values_match(field: str, extracted: int | None, ground_truth: int | None) -> bool:
    """Compare parsed extracted value against parsed ground truth."""
    if extracted is None and ground_truth is None:
        return True
    if extracted is None or ground_truth is None:
        return False
    if field in ("minimum_funding_duration", "maximum_funding_duration"):
        return extracted == ground_truth
    if field == "funding_percentage":
        return extracted == ground_truth
    if ground_truth == 0:
        return extracted == 0
    return abs(extracted - ground_truth) / ground_truth <= 0.01


result = result.with_columns(
    pl.col(field)
    .map_elements(parser, return_dtype=pl.Int64)
    .alias(field)
    for field, parser in FIELD_PARSERS.items()
    if field in result.columns
)
result.select("id", "title", *EXTRACTION_FIELDS)

# Additional Cleaning

In [34]:
result = result.with_columns(
    pl.col("title")
    .str.strip_prefix("Funding opportunity:")
    .str.strip_chars()
)#.rename({"metadata": "metadata_table"})

result.write_ndjson(Path(".").resolve().parent / "data_cache" / "opportunities" / "extracted_opportunities.jsonl")

result

shape: (2_102, 22)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ status ┆ funders    ┆ funding_ty ┆ total_fun ┆ … ┆ minimum_a ┆ minimum_f ┆ maximum_f ┆ funding_p │
│ ---    ┆ ---        ┆ pe         ┆ ding      ┆   ┆ ward      ┆ unding_du ┆ unding_du ┆ ercentage │
│ str    ┆ list[str]  ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ration    ┆ ration    ┆ ---       │
│        ┆            ┆ str        ┆ i64       ┆   ┆ i64       ┆ ---       ┆ ---       ┆ i64       │
│        ┆            ┆            ┆           ┆   ┆           ┆ i64       ┆ i64       ┆           │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ 10000000  ┆ … ┆ 10000000  ┆ null      ┆ null      ┆ null      │
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ 20000000  ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ Closed ┆ ["IUK"]    ┆ Grant      ┆ 1500000   ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ Closed ┆ ["EPSRC"]  ┆ Grant      ┆ 2000000   ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ Closed ┆ ["EPSRC"]  ┆ Grant      ┆ 4000000   ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ …      ┆ …          ┆ …          ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …         │
│ Closed ┆ ["STFC"]   ┆ Grant      ┆ 2000000   ┆ … ┆ null      ┆ null      ┆ 36        ┆ null      │
│ Closed ┆ ["ESRC",   ┆ Grant      ┆ 3000000   ┆ … ┆ 350000    ┆ 36        ┆ 36        ┆ 80        │
│        ┆ "AHRC"]    ┆            ┆           ┆   ┆           ┆           ┆           ┆           │
│ Closed ┆ ["NERC"]   ┆ Grant      ┆ 19000000  ┆ … ┆ 2500000   ┆ null      ┆ 48        ┆ 80        │
│ Closed ┆ ["NERC"]   ┆ Grant      ┆ 24000000  ┆ … ┆ 3000000   ┆ null      ┆ 48        ┆ 80        │
│ Closed ┆ ["NERC"]   ┆ Grant      ┆ 20000000  ┆ … ┆ null      ┆ null      ┆ 48        ┆ 80        │
└────────┴────────────┴────────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘